# Lab 11: Oriented Bounding Box (OBB) Detection

**Duration:** 2 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
A normal detector draws a box that is always straight (the sides are horizontal and vertical). That works for a person standing up. It works badly for a **ship seen from above**, a **plane**, a **long parcel on a belt** or a **line of text**: the straight box is much bigger than the object and it covers the neighbours. An **oriented bounding box** (OBB) is a box that is **rotated** to fit the object, and it also tells you the **angle**. In this lab you run a pretrained OBB model, read the five numbers of a rotated box, and measure how much better the fit is.

**By the end of this lab you can:**
1. Explain the difference between a normal box and an oriented box, and when you need the oriented one.
2. Run a pretrained OBB model and read the output (centre, width, height, angle, class, confidence).
3. Calculate how much empty space a normal box has compared with an oriented box.
4. Show that an OBB model keeps working when the picture is rotated.
5. Explain why this model works on pictures taken from above and not on a normal table-top camera.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. Pictures and live video appear under the cell that makes them. Press **Ctrl + S** often.

> **No training in this lab.** The model was trained by Ultralytics on the DOTA collection of aerial pictures (15 classes). You only use it.

## 2. Background

**Normal box and oriented box.** The detector of Lab 10 gives a box `x1, y1, x2, y2`. The sides are always straight. An OBB model gives a box with **five numbers**: the centre `(x, y)`, the width `w`, the height `h` and the **angle** `r` (rotation). The box turns to fit the object.

| | Normal box (Lab 10) | Oriented box (this lab) |
|---|---|---|
| Numbers | `x1, y1, x2, y2` | `x, y, w, h, angle` |
| Fit on a turned object | Large empty corners | Tight |
| Overlap with the neighbours | High | Low |
| Tells the direction of the object | No | Yes (the angle) |

**Where OBB is used.** Pictures from above (satellite, drone, a camera over a belt or a car park): ships in a harbour, planes at an airport, vehicles, storage tanks, parcels, pieces on a table. Also text lines and documents that are turned.

**The model of this lab.** `yolo26n-obb.pt` was trained on **DOTA**, a collection of aerial pictures. It knows 15 classes: plane, ship, storage tank, baseball diamond, tennis court, basketball court, ground track field, harbor, bridge, large vehicle, small vehicle, helicopter, roundabout, soccer ball field and swimming pool. It is **not** made for a normal camera that looks at a table from the side. If you point it at a cup it finds nothing. This is not a fault: the model never saw cups from the side. To find your own objects you must teach the model, which you do in Lab 33 for normal boxes.

**The angle.** The model gives the angle in **radians**. To get degrees: `degrees = radians * 180 / 3.14159`. A box turned by 90 degrees looks the same as one turned by 0 degrees with width and height swapped, so the model uses a fixed rule for which side is the width.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(11)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (15 min)

### Step 1: A Helper to Show Pictures

In [ ]:
from IPython.display import Image, display

def show(*names, width=420):
    """Show picture files under the cell."""
    for n in names:
        print(n)
        display(Image(filename=n, width=width))

### Step 2: Your Workspace
This notebook lives in the `lab11` folder inside your own folder. Check where you are:

In [ ]:
%%bash
pwd
ls

### Step 3: Check PyTorch and YOLO
If `CUDA available` is `False` or the PyTorch version ends with `+cpu`, stop and call your instructor. Do **not** run `pip install` in this lab.

In [ ]:
import torch
import ultralytics

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
print("Ultralytics version:", ultralytics.__version__)

### Step 4: Get the Model and the Aerial Picture
The cell copies the OBB model and a picture of boats seen from above from the models folder of your own folder (the Lab 0 setup put them there). If the picture is missing, the cell tries to download it once.

In [ ]:
import glob, os, shutil, urllib.request

models = os.path.join(os.environ["LAB_HOME"], "models")
for name in ("yolo26n-obb.pt", "yolo26n.pt", "boats.jpg"):
    source = os.path.join(models, name)
    if os.path.exists(source) and not os.path.exists(name):
        shutil.copy(source, name)
if not os.path.exists("boats.jpg"):
    try:
        request = urllib.request.Request("https://github.com/ultralytics/assets/releases/download/v0.0.0/boats.jpg", headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(request, timeout=30) as reply, open("boats.jpg", "wb") as out:
            out.write(reply.read())
        print("boats.jpg downloaded")
    except Exception as error:
        print("Cannot get boats.jpg:", error)
        print("Ask your instructor for the picture, or copy any picture taken from above into this folder.")

print(sorted(glob.glob("*.jpg")), sorted(glob.glob("*.pt")))
show("boats.jpg", width=520)

---
# Part B: Detect with Oriented Boxes (35 min)

### Step 1: Run the OBB Model
The cell runs the model and prints the **number of objects for each class**, then the five numbers of the first objects. The angle is changed from radians to degrees.

In [ ]:
import math
from collections import Counter

import cv2
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else "cpu"
obb_model = YOLO("yolo26n-obb.pt")


def detect_obb(image, conf=0.25, save_as="obb.jpg", show_rows=6):
    result = obb_model(image, conf=conf, device=device, verbose=False)[0]
    classes = [result.names[int(c)] for c in result.obb.cls]
    print(f"Image: {image}   Confidence limit: {conf}   Objects found: {len(classes)}")
    for name, count in Counter(classes).most_common():
        print(f"  {name:<18} {count}")
    print("First objects: centre (x, y), width, height, angle in degrees")
    for i in range(min(show_rows, len(classes))):
        x, y, w, h, r = result.obb.xywhr[i].tolist()
        print(f"  {classes[i]:<12} conf {float(result.obb.conf[i]):.2f}   ({x:6.0f}, {y:6.0f})   {w:5.0f} x {h:5.0f}   {math.degrees(r):6.1f} deg")
    cv2.imwrite(save_as, result.plot())
    return result


result = detect_obb("boats.jpg")
show("obb.jpg", width=640)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which classes were found, and how many of each?:
- What is the angle of the first object in degrees?:
- Are all boxes turned the same way? Why or why not?:

### Step 2: The Four Corners
A rotated box is easier to draw with its **four corners**. `result.obb.xyxyxyxy` gives the corners of every box (8 numbers: x1, y1 to x4, y4). The cell prints the corners of the first object.

In [ ]:
corners = result.obb.xyxyxyxy[0].tolist()
print("Corners of the first object:")
for k, (x, y) in enumerate(corners, start=1):
    print(f"  corner {k}: ({x:.0f}, {y:.0f})")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Do the corners make a straight box or a turned one?:
- How would you draw this box with cv2.polylines?:

### Step 3: Change the Confidence Limit
The confidence limit works as in Lab 10. Run the model with a low and a high limit and compare the number of objects.

In [ ]:
for limit in (0.10, 0.25, 0.50, 0.75):
    r = obb_model("boats.jpg", conf=limit, device=device, verbose=False)[0]
    print(f"limit {limit:.2f}: {len(r.obb)} objects")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Number of objects at each limit:
- Which limit would you use to count boats in a harbour, and why:

---
# Part C: How Much Better Is the Fit? (30 min)

### Step 1: Compare the Areas
For every object the cell makes the **normal box** that just covers the oriented box (`result.obb.xyxy`). Then it compares the two areas. If the oriented box is only half of the normal box, then half of the normal box is empty space.

In [ ]:
def fit_report(result):
    ratios = []
    for i in range(len(result.obb)):
        x, y, w, h, r = result.obb.xywhr[i].tolist()
        x1, y1, x2, y2 = result.obb.xyxy[i].tolist()
        normal_area = (x2 - x1) * (y2 - y1)
        oriented_area = w * h
        if normal_area > 0:
            ratios.append(oriented_area / normal_area)
    ratios.sort()
    mean = sum(ratios) / len(ratios)
    print(f"Objects: {len(ratios)}")
    print(f"Oriented box is on average {mean * 100:.0f} % of the normal box")
    print(f"Tightest fit: {ratios[0] * 100:.0f} %   Best fit: {ratios[-1] * 100:.0f} %")
    return ratios


ratios = fit_report(result)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Average percentage and what it means:
- Which objects have the worst fit with a normal box (the lowest percentage)?:

### Step 2: Normal Detector or OBB on the Same Picture?
The detector of Lab 10 (`yolo26n.pt`) was trained on COCO. Run it on the same picture and compare. Look at the two pictures.

In [ ]:
det_model = YOLO("yolo26n.pt")
r = det_model("boats.jpg", conf=0.25, device=device, verbose=False)[0]
print("Normal detector (COCO) found", len(r.boxes), "objects:", Counter(r.names[int(c)] for c in r.boxes.cls).most_common(4))
cv2.imwrite("normal.jpg", r.plot())
show("normal.jpg", "obb.jpg", width=480)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many objects did the normal detector find, and how many did the OBB model find?:
- Why is the result so different? Think about the training data of the two models.:
- Which model would you use in a harbour, and which on a shop camera?:

---
# Part D: Does the Angle Change When the Picture Turns? (20 min)

### Step 1: Turn the Picture
The cell turns `boats.jpg` by 0, 30, 60 and 90 degrees (the picture grows so nothing is cut) and counts the objects. A good OBB model should find about the same number of objects in every picture.

In [ ]:
def rotate(img, degrees):
    h, w = img.shape[:2]
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), degrees, 1.0)
    cos, sin = abs(matrix[0, 0]), abs(matrix[0, 1])
    new_w, new_h = int(h * sin + w * cos), int(h * cos + w * sin)
    matrix[0, 2] += new_w / 2 - w / 2
    matrix[1, 2] += new_h / 2 - h / 2
    return cv2.warpAffine(img, matrix, (new_w, new_h), borderValue=(0, 0, 0))


picture = cv2.imread("boats.jpg")
for degrees in (0, 30, 60, 90):
    turned = rotate(picture, degrees)
    cv2.imwrite(f"turned_{degrees}.jpg", turned)
    r = obb_model(turned, conf=0.25, device=device, verbose=False)[0]
    cv2.imwrite(f"turned_{degrees}_obb.jpg", r.plot())
    print(f"turned {degrees:>2} degrees: {len(r.obb)} objects")
show("turned_30_obb.jpg", "turned_60_obb.jpg", width=420)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Number of objects at 0, 30, 60 and 90 degrees:
- Did the angles of the boxes change when the picture turned? Look at the two pictures:
- Why does a normal detector have a harder time with turned objects?:

---
# Part E: Live on the USB Camera (20 min)

The model was trained on pictures from above, so it will not find the objects on your table. To see it work live, **point the camera at a screen** that shows `boats.jpg` (open the picture on a phone or a laptop), or at a printed aerial picture. Run the cell, use the **Confidence** slider, and press **Stop** when you are done.

In [ ]:
import labview


def process(frame, state):
    r = obb_model(frame, conf=state["Confidence"], device=device, verbose=False)[0]
    shown = r.plot()
    counts = Counter(r.names[int(c)] for c in r.obb.cls)
    summary = ", ".join(f"{k}: {v}" for k, v in counts.most_common(3)) or "nothing found"
    cv2.putText(shown, f"{state['fps']:.0f} FPS  conf {state['Confidence']:.2f}", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
    cv2.putText(shown, summary, (10, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    return shown


live = labview.live(process, controls={"Confidence": (0.05, 0.95, 0.05, 0.30)})

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen:
- What happened when you pointed the camera at a normal table scene?:
- What happened when you tilted the screen or the paper?:

---
# Part F: Your Own Test (15 min)
Choose **one** and do it:

- Find another aerial picture (a map screenshot of a car park, a harbour, an airport) and run `detect_obb` on it. Save it in this folder first.
- Find the confidence limit that counts the boats best, and compare the count with your own count by eye.
- Write a function that prints only the objects whose angle is more than 45 degrees.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- What you chose:
- Result and what you learned:

> **Where next.** In Lab 12 you use a model that finds the **exact shape** of an object (segmentation), and you use zones. In Lab 33 you will teach a model objects that you choose yourself.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The OBB model runs on `boats.jpg` and the five numbers of a box are explained | |
| 2 | The angle is changed from radians to degrees and the corners are printed | |
| 3 | The area comparison is done and you explain what the percentage means | |
| 4 | The normal detector and the OBB model are compared on the same picture | |
| 5 | The rotation test is done for four angles and you explain the result | |
| 6 | The live view runs on a screen or a printed aerial picture | |
| 7 | Your own test of Part F is done | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers**

- 1. What is the difference between a normal box and an oriented box? Name the five numbers of an oriented box.:
- 2. Why is a normal box a bad fit for a ship seen from above?:
- 3. The angle of a box is 1.57 radians. How many degrees is that?:
- 4. The oriented box has 60 % of the area of the normal box. What does this tell you?:
- 5. Why does the OBB model find nothing on a cup on your desk?:
- 6. Name two jobs where an oriented box helps more than a normal box.:
- 7. The count of objects stays the same when the picture is turned. What does this show about the model?:
- 8. How could you get an OBB model that finds your own objects on a desk? (Hint: Lab 33.):

## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `CUDA available: False` | The PyTorch of your environment is the CPU version | Stop and call your instructor. Do not run `pip install`. |
| `yolo26n-obb.pt` is downloaded in the middle of the lab | The Lab 0 setup did not download it | Ask your teacher. The lab works if the Jetson has internet. |
| `Cannot get boats.jpg` | No internet and the picture is not in the models folder | Ask your instructor for the picture, or use any aerial picture you have |
| Nothing is found on the live camera | The model needs pictures from above | Point the camera at a screen that shows `boats.jpg` |
| `Cannot open the camera` | Another program uses the camera | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| `CUDA out of memory` | Other notebooks use the shared 8 GB | Press Stop, shut down other kernels (Kernel, Shut Down All Kernels) and run again |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(11)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Ultralytics, Oriented Bounding Box (OBB) Detection: https://docs.ultralytics.com/tasks/obb/
- DOTA dataset: https://captain-whu.github.io/DOTA/